##  config

In [ ]:
import os
import sys
from pathlib import Path

ROOT = "/home/wangxc1117/STDK_GNA_Research"
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import torch
import optuna
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter

torch.set_default_dtype(torch.float32)
optuna.logging.set_verbosity(optuna.logging.WARNING)

from examples.baselines.stdk.st_interp import create_model
from spatial_adapter.models.spatial_adapter import (
    SpatialAdapterConfig,
    ADMMConfig,
    TrainingConfig,
    BasisConfig,
)
from examples.baselines.timesplit.experiment_core import (
    seed_everything,
    build_fixed_location_subset,
    build_contiguous_time_splits,
    build_stdk_model_config,
    train_simple_loop,
    predict_all_simple,
    rmse_pooled,
    rmse_on_time_subset,
    DictDataset,
    collate_fn,
    new_trend_basis,
    fit_adapter_reconstruct_all_times,
    plot_trial_maps,
    fmt_pm,
    build_standard_bin_edges,
    sv_match_loss_for_prediction_matrix,
)


# Global settings & dirs
SEED = 123

WEATHER2K_NPY = Path("/home/wangxc1117/Weather2K/weather2k.npy")
TARGET_VAR_IDX = 4
LAT_IDX = 0
LON_IDX = 1
T_KEEP = 1000

EPOCHS = 350
BATCH_SIZE = 512
LR = 1e-3
WEIGHT_DECAY = 1e-5

SPACE_RATIO_KEEP = 0.1

TRAIN_RATIO_TIME = 0.1
VAL_RATIO_TIME = 0.1
TEST_RATIO_TIME = 0.8

GNA_BATCH_SIZE = 64
N_TRIALS = 50
TAU_MIN = 1e-8
TAU_MAX = 1e4

K_FIXED = 40
N_RUNS_FIXED_K = 100

SEMIVAR_WEIGHTED = True
SEMIVAR_NORMALIZED = False
SEMIVAR_ESTIMATOR = "matheron"

TUNING_TARGET = "rmse"

if TUNING_TARGET not in {"rmse", "covfrob", "sv_score"}:
    raise ValueError("TUNING_TARGET must be one of {'rmse', 'covfrob', 'sv_score'}")

RESULT_DIR = Path("./weather2k")
REPEAT_DIR = RESULT_DIR / (
    f"{TUNING_TARGET}_tuning"
    f"/var{TARGET_VAR_IDX}_tkeep{T_KEEP}"
    f"_k_{K_FIXED}_fixedspace{SPACE_RATIO_KEEP}"
    f"_time_train{TRAIN_RATIO_TIME}_val{VAL_RATIO_TIME}_test{TEST_RATIO_TIME}"
)
REPEAT_DIR.mkdir(parents=True, exist_ok=True)

SUMMARY_CSV = REPEAT_DIR / f"k_{K_FIXED}_repeat_runs_summary.csv"

PRED_DIR = REPEAT_DIR / "saved_predictions"
PRED_DIR.mkdir(parents=True, exist_ok=True)

TRIAL_DIR = REPEAT_DIR / "trial_results"
TRIAL_DIR.mkdir(parents=True, exist_ok=True)

PHI_DIR = REPEAT_DIR / "saved_phi"
PHI_DIR.mkdir(parents=True, exist_ok=True)

ALL_TRIAL_CSV = TRIAL_DIR / "all_trial_results.csv"

DIAG_DIR = REPEAT_DIR / "diagnostics"
DIAG_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device =", device, flush=True)

seed_everything(SEED)

config = SpatialAdapterConfig(
    admm=ADMMConfig(
        rho=1.0,
        dual_momentum=0.2,
        max_iters=3000,
        min_outer=20,
        tol=1e-4,
    ),
    training=TrainingConfig(
        lr_mu=1e-2,
        batch_size=GNA_BATCH_SIZE,
        pretrain_epochs=5,
    ),
    basis=BasisConfig(
        phi_every=5,
        phi_freeze=200,
    ),
)


# Data helpers
def load_weather2k_as_long_df(
    npy_path,
    target_var_idx,
    lat_idx,
    lon_idx,
    t_keep=None,
    normalize_xy=True,
):
    arr = np.load(str(npy_path)).astype(np.float32)

    if arr.ndim != 3:
        raise ValueError(f"Expected arr.ndim == 3 (S, V, T), got shape={arr.shape}")

    n_sites, n_vars, n_times_full = arr.shape

    if not (0 <= lat_idx < n_vars and 0 <= lon_idx < n_vars and 0 <= target_var_idx < n_vars):
        raise ValueError(
            f"Bad index: lat_idx={lat_idx}, lon_idx={lon_idx}, "
            f"target_var_idx={target_var_idx}, n_vars={n_vars}"
        )

    lat = arr[:, lat_idx, 0].astype(np.float32)
    lon = arr[:, lon_idx, 0].astype(np.float32)
    z_full = arr[:, target_var_idx, :].astype(np.float32)

    if t_keep is not None:
        if t_keep <= 0 or t_keep > n_times_full:
            raise ValueError(f"t_keep must be in [1, {n_times_full}], got {t_keep}")
        z_use = z_full[:, -t_keep:]
        n_times = t_keep
    else:
        z_use = z_full
        n_times = n_times_full

    if normalize_xy:
        lon_min, lon_max = float(np.min(lon)), float(np.max(lon))
        lat_min, lat_max = float(np.min(lat)), float(np.max(lat))
        x = ((lon - lon_min) / (lon_max - lon_min + 1e-12)).astype(np.float32)
        y = ((lat - lat_min) / (lat_max - lat_min + 1e-12)).astype(np.float32)
    else:
        x = lon.astype(np.float32)
        y = lat.astype(np.float32)

    t = np.arange(n_times, dtype=np.int64)

    xx = np.repeat(x, n_times)
    yy = np.repeat(y, n_times)
    tt = np.tile(t, n_sites)
    zz = z_use.reshape(-1).astype(np.float32)

    df = pd.DataFrame({
        "x": xx,
        "y": yy,
        "t": tt,
        "z": zz,
    })

    z_np = df["z"].to_numpy(np.float32)
    ok = np.isfinite(z_np)
    df = df.loc[ok].reset_index(drop=True)

    return df


# Metric helpers
# Objective helpers
def choose_objective_value(val_rmse, covfrob_reg_val, sv_loss_val):
    if TUNING_TARGET == "rmse":
        return float(val_rmse)
    if TUNING_TARGET == "covfrob":
        return float(covfrob_reg_val)
    if TUNING_TARGET == "sv_score":
        return float(sv_loss_val)
    raise ValueError(f"Unknown TUNING_TARGET: {TUNING_TARGET}")


def objective_label():
    if TUNING_TARGET == "rmse":
        return "best_val_rmse"
    if TUNING_TARGET == "covfrob":
        return "best_val_covfrob"
    if TUNING_TARGET == "sv_score":
        return "best_val_sv_score"
    raise ValueError(f"Unknown TUNING_TARGET: {TUNING_TARGET}")


def objective_best_by_column():
    if TUNING_TARGET == "rmse":
        return "val_rmse_raw"
    if TUNING_TARGET == "covfrob":
        return "covfrob_reg_val"
    if TUNING_TARGET == "sv_score":
        return "sv_loss_val"
    raise ValueError(f"Unknown TUNING_TARGET: {TUNING_TARGET}")


# IO helpers
def load_all_trials_or_empty():
    if ALL_TRIAL_CSV.exists():
        return pd.read_csv(ALL_TRIAL_CSV)
    return pd.DataFrame()


def load_summary_or_empty():
    if SUMMARY_CSV.exists():
        return pd.read_csv(SUMMARY_CSV)
    return pd.DataFrame()


def list_prediction_npz_files():
    return sorted(PRED_DIR.glob("seed_*.npz"))


def print_main_paths():
    print("REPEAT_DIR:", REPEAT_DIR, flush=True)
    print("SUMMARY_CSV:", SUMMARY_CSV, flush=True)
    print("ALL_TRIAL_CSV:", ALL_TRIAL_CSV, flush=True)
    print("PRED_DIR:", PRED_DIR, flush=True)
    print("TRIAL_DIR:", TRIAL_DIR, flush=True)
    print("PHI_DIR:", PHI_DIR, flush=True)
    print("DIAG_DIR:", DIAG_DIR, flush=True)

In [ ]:
from spatial_adapter.metrics import rmse_pooled, mae_pooled, r2_pooled, empirical_cov, cov_frob_observed


## seed runs

In [ ]:
def run_once_fixed_k(run_seed: int):
    seed_everything(run_seed)

    df_full = load_weather2k_as_long_df(
        npy_path=WEATHER2K_NPY,
        target_var_idx=TARGET_VAR_IDX,
        lat_idx=LAT_IDX,
        lon_idx=LON_IDX,
        t_keep=T_KEEP,
        normalize_xy=True,
    )

    df_run, keep_sites_run, n_sites_full_run = build_fixed_location_subset(
        df=df_full,
        keep_ratio=SPACE_RATIO_KEEP,
        seed=run_seed + 11111,
    )

    df_run["t_norm"] = (
        (df_run["t"] - df_run["t"].min())
        / (df_run["t"].max() - df_run["t"].min() + 1e-12)
    ).astype(np.float32)

    (
        train_mask_flat_run,
        val_mask_flat_run,
        test_mask_flat_run,
        train_time_idx_run,
        val_time_idx_run,
        test_time_idx_run,
        uniq_t_run,
        n_times_run,
    ) = build_contiguous_time_splits(
        df=df_run,
        train_ratio=TRAIN_RATIO_TIME,
        val_ratio=VAL_RATIO_TIME,
        test_ratio=TEST_RATIO_TIME,
    )

    df_run["z_raw"] = df_run["z"].astype(np.float32)

    z_train_raw = df_run.loc[train_mask_flat_run, "z_raw"].to_numpy(np.float32)
    z_mean_run = float(np.mean(z_train_raw))
    z_sd_run = float(np.std(z_train_raw, ddof=0))
    if z_sd_run < 1e-12:
        z_sd_run = 1.0

    df_run["z"] = (
        (df_run["z_raw"].to_numpy(np.float32) - z_mean_run) / (z_sd_run + 1e-12)
    ).astype(np.float32)

    def to_raw(arr_std: np.ndarray) -> np.ndarray:
        return arr_std * z_sd_run + z_mean_run

    coords_all_run = df_run[["x", "y"]].to_numpy(np.float32)
    t_all_run = df_run["t_norm"].to_numpy(np.float32).reshape(-1, 1)
    y_all_run = df_run["z"].to_numpy(np.float32).reshape(-1, 1)
    X_all_run = np.empty((df_run.shape[0], 0), dtype=np.float32)

    X_train_run = X_all_run[train_mask_flat_run]
    coords_train_run = coords_all_run[train_mask_flat_run]
    t_train_run = t_all_run[train_mask_flat_run]
    y_train_run = y_all_run[train_mask_flat_run]

    train_dataset_run = DictDataset(
        torch.from_numpy(X_train_run),
        torch.from_numpy(coords_train_run),
        torch.from_numpy(t_train_run),
        torch.from_numpy(y_train_run),
    )

    g = torch.Generator()
    g.manual_seed(run_seed + 1000)

    train_loader_run = DataLoader(
        train_dataset_run,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=g,
        num_workers=0,
        pin_memory=True,
        collate_fn=collate_fn,
    )

    stdk_config = build_stdk_model_config(
        EPOCHS=EPOCHS,
        LR=LR,
        WEIGHT_DECAY=WEIGHT_DECAY,
        BATCH_SIZE=BATCH_SIZE,
    )

    stdk_run = create_model(
        stdk_config,
        train_coords=coords_train_run,
    ).to(device)

    stdk_run = train_simple_loop(
        model=stdk_run,
        train_loader=train_loader_run,
        device=device,
        config=stdk_config,
    )

    y_hat_all_run = predict_all_simple(
        model=stdk_run,
        X=torch.from_numpy(X_all_run),
        coords=torch.from_numpy(coords_all_run),
        t=torch.from_numpy(t_all_run),
        batch_size=BATCH_SIZE,
        device=device,
    )

    locs_run, inv_loc_run = np.unique(coords_all_run, axis=0, return_inverse=True)
    t_to_idx_run = {t: i for i, t in enumerate(uniq_t_run)}

    T_run = len(uniq_t_run)
    N_run = len(locs_run)

    t_idx_run = np.array([t_to_idx_run[t] for t in df_run["t"].to_numpy()])
    s_idx_run = inv_loc_run

    y_stdk_run = np.full((T_run, N_run), np.nan, np.float32)
    y_true_run = np.full((T_run, N_run), np.nan, np.float32)

    y_stdk_run[t_idx_run, s_idx_run] = y_hat_all_run
    y_true_run[t_idx_run, s_idx_run] = df_run["z"].to_numpy(np.float32)

    y_stdk_raw_run = to_raw(y_stdk_run)
    y_true_raw_run = to_raw(y_true_run)

    residual_true_run = y_true_run - y_stdk_run

    time_feat_run = (
        (uniq_t_run - uniq_t_run.min())
        / (uniq_t_run.max() - uniq_t_run.min() + 1e-12)
    ).astype(np.float32)

    cont_all_run = (
        torch.from_numpy(time_feat_run)
        .float()
        .unsqueeze(1)
        .repeat(1, N_run)
        .unsqueeze(-1)
    )

    train_cont_train_run = cont_all_run[train_time_idx_run]
    train_y_train_run = torch.from_numpy(
        residual_true_run[train_time_idx_run, :]
    ).float()
    train_idx_train_run = torch.arange(len(train_time_idx_run), dtype=torch.long)

    gna_loader_train_run = DataLoader(
        TensorDataset(train_idx_train_run, train_cont_train_run, train_y_train_run),
        batch_size=min(GNA_BATCH_SIZE, len(train_time_idx_run)),
        shuffle=True,
        drop_last=False,
    )

    residual_true_tensor_all_run = torch.from_numpy(residual_true_run).float()

    seed_phi_dir = PHI_DIR / f"seed_{run_seed}"
    seed_phi_dir.mkdir(parents=True, exist_ok=True)

    semivar_bin_edges = build_standard_bin_edges(locs_run)
    full_time_idx_run = np.arange(T_run, dtype=int)

    def rmse_std(y_pred_std: np.ndarray, time_idx: np.ndarray) -> float:
        mask = np.isfinite(y_true_run[time_idx, :]) & np.isfinite(y_pred_std[time_idx, :])
        return rmse_pooled(y_true_run[time_idx, :], y_pred_std[time_idx, :], mask)

    def rmse_raw(y_pred_std: np.ndarray, time_idx: np.ndarray) -> float:
        y_pred_raw = to_raw(y_pred_std[time_idx, :])
        mask = np.isfinite(y_true_raw_run[time_idx, :]) & np.isfinite(y_pred_raw)
        return rmse_pooled(y_true_raw_run[time_idx, :], y_pred_raw, mask)

    def covfrob_std(y_pred_std: np.ndarray, time_idx: np.ndarray) -> float:
        return cov_frob_observed(
            y_true_run[time_idx, :],
            y_pred_std[time_idx, :],
        )

    def covfrob_raw(y_pred_std: np.ndarray, time_idx: np.ndarray) -> float:
        return cov_frob_observed(
            y_true_raw_run[time_idx, :],
            to_raw(y_pred_std[time_idx, :]),
        )

    def sv_loss_std(y_pred_std: np.ndarray, time_idx: np.ndarray) -> float:
        out = sv_match_loss_for_prediction_matrix(
            coords=locs_run,
            y_true=y_true_run,
            y_pred=y_pred_std,
            time_idx=time_idx,
            bin_edges=semivar_bin_edges,
            estimator=SEMIVAR_ESTIMATOR,
            weighted=SEMIVAR_WEIGHTED,
            normalized=SEMIVAR_NORMALIZED,
        )
        return float(out["loss"])

    def sv_loss_raw(y_pred_std: np.ndarray, time_idx: np.ndarray) -> float:
        out = sv_match_loss_for_prediction_matrix(
            coords=locs_run,
            y_true=y_true_raw_run,
            y_pred=to_raw(y_pred_std),
            time_idx=time_idx,
            bin_edges=semivar_bin_edges,
            estimator=SEMIVAR_ESTIMATOR,
            weighted=SEMIVAR_WEIGHTED,
            normalized=SEMIVAR_NORMALIZED,
        )
        return float(out["loss"])

    writer_unreg = SummaryWriter(
        str(REPEAT_DIR / "logs_unreg" / f"seed_{run_seed}" / "unreg_tau1_0_tau2_0")
    )

    trend_unreg, basis_unreg = new_trend_basis(N_run, K_FIXED)
    fit_unreg = fit_adapter_reconstruct_all_times(
        tag="unreg_tau1_0_tau2_0",
        tau1=0.0,
        tau2=0.0,
        trend=trend_unreg,
        basis=basis_unreg,
        train_loader=gna_loader_train_run,
        val_cont=train_cont_train_run,
        val_y=train_y_train_run,
        locs=locs_run,
        config=config,
        device=device,
        cont_all=cont_all_run,
        residual_true_tensor_all=residual_true_tensor_all_run,
        writer=writer_unreg,
    )
    writer_unreg.close()

    residual_full_unreg_run = fit_unreg["pred_all"]
    diag_train_unreg_run = fit_unreg["diag_train"]
    diag_all_unreg_run = fit_unreg["diag_all"]
    phi_unreg_run = fit_unreg["phi"]

    np.savez_compressed(
        seed_phi_dir / "unreg_phi.npz",
        phi=phi_unreg_run.astype(np.float32),
        tau1=np.array([0.0], dtype=np.float64),
        tau2=np.array([0.0], dtype=np.float64),
        seed=np.array([run_seed], dtype=np.int32),
    )

    y_final_unreg_run = y_stdk_run + residual_full_unreg_run
    y_final_unreg_raw_run = to_raw(y_final_unreg_run)

    trial_rows = []
    trial_cache = {}

    def objective(trial: optuna.Trial):
        tau1 = trial.suggest_float("tau1", TAU_MIN, TAU_MAX, log=True)
        tau2 = trial.suggest_float("tau2", TAU_MIN, TAU_MAX, log=True)

        tag = f"trial_{trial.number:03d}_tau1_{tau1:.3e}_tau2_{tau2:.3e}"
        writer_trial = SummaryWriter(
            str(REPEAT_DIR / "logs_reg" / f"seed_{run_seed}" / tag)
        )

        trend_trial, basis_trial = new_trend_basis(N_run, K_FIXED)
        fit_trial = fit_adapter_reconstruct_all_times(
            tag=tag,
            tau1=tau1,
            tau2=tau2,
            trend=trend_trial,
            basis=basis_trial,
            train_loader=gna_loader_train_run,
            val_cont=train_cont_train_run,
            val_y=train_y_train_run,
            locs=locs_run,
            config=config,
            device=device,
            cont_all=cont_all_run,
            residual_true_tensor_all=residual_true_tensor_all_run,
            writer=writer_trial,
        )
        writer_trial.close()

        residual_full_trial = fit_trial["pred_all"]
        diag_train_trial = fit_trial["diag_train"]
        diag_all_trial = fit_trial["diag_all"]
        phi_trial = fit_trial["phi"]

        y_final_trial = y_stdk_run + residual_full_trial

        train_rmse_std = rmse_std(y_final_trial, train_time_idx_run)
        val_rmse_std = rmse_std(y_final_trial, val_time_idx_run)
        test_rmse_std = rmse_std(y_final_trial, test_time_idx_run)
        full_rmse_std = rmse_std(y_final_trial, full_time_idx_run)

        train_rmse_raw = rmse_raw(y_final_trial, train_time_idx_run)
        val_rmse_raw = rmse_raw(y_final_trial, val_time_idx_run)
        test_rmse_raw = rmse_raw(y_final_trial, test_time_idx_run)
        full_rmse_raw = rmse_raw(y_final_trial, full_time_idx_run)

        covfrob_reg_train = covfrob_std(y_final_trial, train_time_idx_run)
        covfrob_reg_val = covfrob_std(y_final_trial, val_time_idx_run)
        covfrob_reg_test = covfrob_std(y_final_trial, test_time_idx_run)
        covfrob_reg_full = covfrob_std(y_final_trial, full_time_idx_run)

        covfrob_reg_train_raw = covfrob_raw(y_final_trial, train_time_idx_run)
        covfrob_reg_val_raw = covfrob_raw(y_final_trial, val_time_idx_run)
        covfrob_reg_test_raw = covfrob_raw(y_final_trial, test_time_idx_run)
        covfrob_reg_full_raw = covfrob_raw(y_final_trial, full_time_idx_run)

        sv_loss_train = sv_loss_std(y_final_trial, train_time_idx_run)
        sv_loss_val = sv_loss_std(y_final_trial, val_time_idx_run)
        sv_loss_test = sv_loss_std(y_final_trial, test_time_idx_run)
        sv_loss_full = sv_loss_std(y_final_trial, full_time_idx_run)

        sv_loss_train_raw = sv_loss_raw(y_final_trial, train_time_idx_run)
        sv_loss_val_raw = sv_loss_raw(y_final_trial, val_time_idx_run)
        sv_loss_test_raw = sv_loss_raw(y_final_trial, test_time_idx_run)
        sv_loss_full_raw = sv_loss_raw(y_final_trial, full_time_idx_run)

        objective_value = choose_objective_value(
            val_rmse=val_rmse_raw,
            covfrob_reg_val=covfrob_reg_val,
            sv_loss_val=sv_loss_val,
        )

        phi_path = seed_phi_dir / f"trial_{trial.number:03d}_phi.npz"
        np.savez_compressed(
            phi_path,
            phi=phi_trial.astype(np.float32),
            tau1=np.array([tau1], dtype=np.float64),
            tau2=np.array([tau2], dtype=np.float64),
            seed=np.array([run_seed], dtype=np.int32),
            trial=np.array([trial.number], dtype=np.int32),
        )

        row = {
            "seed": int(run_seed),
            "trial": int(trial.number),
            "tau1": float(tau1),
            "tau2": float(tau2),
            "log10_tau1": float(np.log10(tau1)),
            "log10_tau2": float(np.log10(tau2)),
            "objective_value": float(objective_value),
            "train_rmse": float(train_rmse_std),
            "val_rmse": float(val_rmse_std),
            "test_rmse": float(test_rmse_std),
            "full_rmse": float(full_rmse_std),
            "train_rmse_raw": float(train_rmse_raw),
            "val_rmse_raw": float(val_rmse_raw),
            "test_rmse_raw": float(test_rmse_raw),
            "full_rmse_raw": float(full_rmse_raw),
            "sv_loss_train": float(sv_loss_train),
            "sv_loss_val": float(sv_loss_val),
            "sv_loss_test": float(sv_loss_test),
            "sv_loss_full": float(sv_loss_full),
            "sv_loss_train_raw": float(sv_loss_train_raw),
            "sv_loss_val_raw": float(sv_loss_val_raw),
            "sv_loss_test_raw": float(sv_loss_test_raw),
            "sv_loss_full_raw": float(sv_loss_full_raw),
            "covfrob_reg_train": float(covfrob_reg_train),
            "covfrob_reg_val": float(covfrob_reg_val),
            "covfrob_reg_test": float(covfrob_reg_test),
            "covfrob_reg_full": float(covfrob_reg_full),
            "covfrob_reg_train_raw": float(covfrob_reg_train_raw),
            "covfrob_reg_val_raw": float(covfrob_reg_val_raw),
            "covfrob_reg_test_raw": float(covfrob_reg_test_raw),
            "covfrob_reg_full_raw": float(covfrob_reg_full_raw),
            "recon_mse_train": float(diag_train_trial["recon_mse"]),
            "smooth_penalty_train": float(diag_train_trial["smooth_penalty"]),
            "l1_penalty_train": float(diag_train_trial["l1_penalty"]),
            "total_surrogate_train": float(diag_train_trial["total_surrogate"]),
            "smooth_penalty_per_entry_train": float(diag_train_trial["smooth_penalty_per_entry"]),
            "l1_penalty_per_entry_train": float(diag_train_trial["l1_penalty_per_entry"]),
            "smooth_over_recon_train": float(diag_train_trial["smooth_over_recon"]),
            "l1_over_recon_train": float(diag_train_trial["l1_over_recon"]),
            "recon_mse_all": float(diag_all_trial["recon_mse"]),
            "smooth_penalty_all": float(diag_all_trial["smooth_penalty"]),
            "l1_penalty_all": float(diag_all_trial["l1_penalty"]),
            "total_surrogate_all": float(diag_all_trial["total_surrogate"]),
            "smooth_penalty_per_entry_all": float(diag_all_trial["smooth_penalty_per_entry"]),
            "l1_penalty_per_entry_all": float(diag_all_trial["l1_penalty_per_entry"]),
            "smooth_over_recon_all": float(diag_all_trial["smooth_over_recon"]),
            "l1_over_recon_all": float(diag_all_trial["l1_over_recon"]),
            "phi_path": str(phi_path),
        }

        trial_rows.append(row)
        trial_cache[int(trial.number)] = {
            "tau1": float(tau1),
            "tau2": float(tau2),
            "objective_value": float(objective_value),
            "pred_all": residual_full_trial.astype(np.float32),
            "phi": phi_trial.astype(np.float32),
            "diag_train": diag_train_trial,
            "diag_all": diag_all_trial,
        }

        print(
            f"[seed {run_seed}] trial {trial.number + 1:03d}/{N_TRIALS:03d} | "
            f"target={TUNING_TARGET} | "
            f"tau1={tau1:.3e} | tau2={tau2:.3e} | "
            f"train_rmse={train_rmse_std:.6f} | "
            f"val_rmse={val_rmse_std:.6f} | "
            f"sv_loss_val={sv_loss_val:.6f} | "
            f"covfrob_val={covfrob_reg_val:.6f} | "
            f"objective={objective_value:.6f}",
            flush=True,
        )

        return float(objective_value)

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

    trial_df_run = pd.DataFrame(trial_rows)
    trial_csv_run = TRIAL_DIR / f"seed_{run_seed}_trials.csv"
    trial_df_run.to_csv(trial_csv_run, index=False)

    best_trial_no = int(study.best_trial.number)
    best_trial = trial_cache[best_trial_no]

    best_tau1_run = float(best_trial["tau1"])
    best_tau2_run = float(best_trial["tau2"])
    best_objective_run = float(best_trial["objective_value"])
    residual_full_reg_best_run = best_trial["pred_all"]
    phi_reg_best_run = best_trial["phi"]
    diag_train_reg_best_run = best_trial["diag_train"]
    diag_all_reg_best_run = best_trial["diag_all"]

    np.savez_compressed(
        seed_phi_dir / "reg_best_phi.npz",
        phi=phi_reg_best_run.astype(np.float32),
        tau1=np.array([best_tau1_run], dtype=np.float64),
        tau2=np.array([best_tau2_run], dtype=np.float64),
        seed=np.array([run_seed], dtype=np.int32),
        trial=np.array([best_trial_no], dtype=np.int32),
    )

    y_final_reg_best_run = y_stdk_run + residual_full_reg_best_run
    y_final_reg_best_raw_run = to_raw(y_final_reg_best_run)

    def collect_metrics(prefix: str, y_pred_std: np.ndarray):
        return {
            f"{prefix}_rmse_train": rmse_std(y_pred_std, train_time_idx_run),
            f"{prefix}_rmse_val": rmse_std(y_pred_std, val_time_idx_run),
            f"{prefix}_rmse_test": rmse_std(y_pred_std, test_time_idx_run),
            f"{prefix}_rmse_full": rmse_std(y_pred_std, full_time_idx_run),
            f"{prefix}_rmse_train_raw": rmse_raw(y_pred_std, train_time_idx_run),
            f"{prefix}_rmse_val_raw": rmse_raw(y_pred_std, val_time_idx_run),
            f"{prefix}_rmse_test_raw": rmse_raw(y_pred_std, test_time_idx_run),
            f"{prefix}_rmse_full_raw": rmse_raw(y_pred_std, full_time_idx_run),
            f"{prefix}_covfrob_train": covfrob_std(y_pred_std, train_time_idx_run),
            f"{prefix}_covfrob_val": covfrob_std(y_pred_std, val_time_idx_run),
            f"{prefix}_covfrob_test": covfrob_std(y_pred_std, test_time_idx_run),
            f"{prefix}_covfrob_full": covfrob_std(y_pred_std, full_time_idx_run),
            f"{prefix}_covfrob_train_raw": covfrob_raw(y_pred_std, train_time_idx_run),
            f"{prefix}_covfrob_val_raw": covfrob_raw(y_pred_std, val_time_idx_run),
            f"{prefix}_covfrob_test_raw": covfrob_raw(y_pred_std, test_time_idx_run),
            f"{prefix}_covfrob_full_raw": covfrob_raw(y_pred_std, full_time_idx_run),
            f"{prefix}_sv_loss_train": sv_loss_std(y_pred_std, train_time_idx_run),
            f"{prefix}_sv_loss_val": sv_loss_std(y_pred_std, val_time_idx_run),
            f"{prefix}_sv_loss_test": sv_loss_std(y_pred_std, test_time_idx_run),
            f"{prefix}_sv_loss_full": sv_loss_std(y_pred_std, full_time_idx_run),
            f"{prefix}_sv_loss_train_raw": sv_loss_raw(y_pred_std, train_time_idx_run),
            f"{prefix}_sv_loss_val_raw": sv_loss_raw(y_pred_std, val_time_idx_run),
            f"{prefix}_sv_loss_test_raw": sv_loss_raw(y_pred_std, test_time_idx_run),
            f"{prefix}_sv_loss_full_raw": sv_loss_raw(y_pred_std, full_time_idx_run),
        }

    summary_row = {
        "seed": int(run_seed),
        "n_sites_full": int(n_sites_full_run),
        "n_sites_keep": int(len(keep_sites_run)),
        "n_times": int(n_times_run),
        "n_train_times": int(len(train_time_idx_run)),
        "n_val_times": int(len(val_time_idx_run)),
        "n_test_times": int(len(test_time_idx_run)),
        "z_mean_train_raw": float(z_mean_run),
        "z_sd_train_raw": float(z_sd_run),
        "best_tau1": float(best_tau1_run),
        "best_tau2": float(best_tau2_run),
        "best_trial": int(best_trial_no),
        objective_label(): float(best_objective_run),
        "unreg_recon_mse_train": float(diag_train_unreg_run["recon_mse"]),
        "unreg_smooth_penalty_train": float(diag_train_unreg_run["smooth_penalty"]),
        "unreg_l1_penalty_train": float(diag_train_unreg_run["l1_penalty"]),
        "unreg_total_surrogate_train": float(diag_train_unreg_run["total_surrogate"]),
        "unreg_smooth_penalty_per_entry_train": float(diag_train_unreg_run["smooth_penalty_per_entry"]),
        "unreg_l1_penalty_per_entry_train": float(diag_train_unreg_run["l1_penalty_per_entry"]),
        "unreg_smooth_over_recon_train": float(diag_train_unreg_run["smooth_over_recon"]),
        "unreg_l1_over_recon_train": float(diag_train_unreg_run["l1_over_recon"]),
        "unreg_recon_mse_all": float(diag_all_unreg_run["recon_mse"]),
        "unreg_smooth_penalty_all": float(diag_all_unreg_run["smooth_penalty"]),
        "unreg_l1_penalty_all": float(diag_all_unreg_run["l1_penalty"]),
        "unreg_total_surrogate_all": float(diag_all_unreg_run["total_surrogate"]),
        "unreg_smooth_penalty_per_entry_all": float(diag_all_unreg_run["smooth_penalty_per_entry"]),
        "unreg_l1_penalty_per_entry_all": float(diag_all_unreg_run["l1_penalty_per_entry"]),
        "unreg_smooth_over_recon_all": float(diag_all_unreg_run["smooth_over_recon"]),
        "unreg_l1_over_recon_all": float(diag_all_unreg_run["l1_over_recon"]),
        "reg_best_recon_mse_train": float(diag_train_reg_best_run["recon_mse"]),
        "reg_best_smooth_penalty_train": float(diag_train_reg_best_run["smooth_penalty"]),
        "reg_best_l1_penalty_train": float(diag_train_reg_best_run["l1_penalty"]),
        "reg_best_total_surrogate_train": float(diag_train_reg_best_run["total_surrogate"]),
        "reg_best_smooth_penalty_per_entry_train": float(diag_train_reg_best_run["smooth_penalty_per_entry"]),
        "reg_best_l1_penalty_per_entry_train": float(diag_train_reg_best_run["l1_penalty_per_entry"]),
        "reg_best_smooth_over_recon_train": float(diag_train_reg_best_run["smooth_over_recon"]),
        "reg_best_l1_over_recon_train": float(diag_train_reg_best_run["l1_over_recon"]),
        "reg_best_recon_mse_all": float(diag_all_reg_best_run["recon_mse"]),
        "reg_best_smooth_penalty_all": float(diag_all_reg_best_run["smooth_penalty"]),
        "reg_best_l1_penalty_all": float(diag_all_reg_best_run["l1_penalty"]),
        "reg_best_total_surrogate_all": float(diag_all_reg_best_run["total_surrogate"]),
        "reg_best_smooth_penalty_per_entry_all": float(diag_all_reg_best_run["smooth_penalty_per_entry"]),
        "reg_best_l1_penalty_per_entry_all": float(diag_all_reg_best_run["l1_penalty_per_entry"]),
        "reg_best_smooth_over_recon_all": float(diag_all_reg_best_run["smooth_over_recon"]),
        "reg_best_l1_over_recon_all": float(diag_all_reg_best_run["l1_over_recon"]),
    }

    summary_row.update(collect_metrics("stdk", y_stdk_run))
    summary_row.update(collect_metrics("unreg", y_final_unreg_run))
    summary_row.update(collect_metrics("reg_best", y_final_reg_best_run))

    np.savez_compressed(
        PRED_DIR / f"seed_{run_seed}.npz",
        seed=np.array([run_seed], dtype=np.int32),
        keep_sites_run=np.asarray(keep_sites_run, dtype=np.int32),
        n_sites_full_run=np.array([n_sites_full_run], dtype=np.int32),
        uniq_t_run=np.asarray(uniq_t_run, dtype=np.float32),
        train_time_idx_run=np.asarray(train_time_idx_run, dtype=np.int32),
        val_time_idx_run=np.asarray(val_time_idx_run, dtype=np.int32),
        test_time_idx_run=np.asarray(test_time_idx_run, dtype=np.int32),
        locs_run=locs_run.astype(np.float32),
        y_true_run=y_true_run.astype(np.float32),
        y_true_raw_run=y_true_raw_run.astype(np.float32),
        y_stdk_run=y_stdk_run.astype(np.float32),
        y_stdk_raw_run=y_stdk_raw_run.astype(np.float32),
        y_final_unreg_run=y_final_unreg_run.astype(np.float32),
        y_final_unreg_raw_run=y_final_unreg_raw_run.astype(np.float32),
        y_final_reg_best_run=y_final_reg_best_run.astype(np.float32),
        y_final_reg_best_raw_run=y_final_reg_best_raw_run.astype(np.float32),
        residual_full_unreg_run=residual_full_unreg_run.astype(np.float32),
        residual_full_reg_best_run=residual_full_reg_best_run.astype(np.float32),
        z_mean_run=np.array([z_mean_run], dtype=np.float32),
        z_sd_run=np.array([z_sd_run], dtype=np.float32),
        best_tau1=np.array([best_tau1_run], dtype=np.float64),
        best_tau2=np.array([best_tau2_run], dtype=np.float64),
        best_trial=np.array([best_trial_no], dtype=np.int32),
    )

    print(
        f"[seed {run_seed}] best trial={best_trial_no} | "
        f"tau1={best_tau1_run:.3e} | tau2={best_tau2_run:.3e} | "
        f"test_rmse={summary_row['reg_best_rmse_test']:.6f} | "
        f"test_covfrob={summary_row['reg_best_covfrob_test']:.6f} | "
        f"test_sv={summary_row['reg_best_sv_loss_test']:.6f}",
        flush=True,
    )

    return summary_row

## run

In [ ]:
rows_fixed_k = []

for r in range(N_RUNS_FIXED_K):
    run_seed = SEED + r * 1000
    print(
        f"\n===== FIXED K RUN {r + 1}/{N_RUNS_FIXED_K} | "
        f"seed={run_seed} | target={TUNING_TARGET} | "
        f"space_keep={SPACE_RATIO_KEEP} | "
        f"time_train={TRAIN_RATIO_TIME} | "
        f"time_val={VAL_RATIO_TIME} | "
        f"time_test={TEST_RATIO_TIME} =====",
        flush=True,
    )
    rows_fixed_k.append(run_once_fixed_k(run_seed))

results_fixed_k_df = pd.DataFrame(rows_fixed_k)
results_fixed_k_df.to_csv(SUMMARY_CSV, index=False)

trial_files = sorted(TRIAL_DIR.glob("seed_*_trials.csv"))
if len(trial_files) > 0:
    all_trial_df = pd.concat(
        [pd.read_csv(f) for f in trial_files],
        ignore_index=True,
    )
    all_trial_df.to_csv(ALL_TRIAL_CSV, index=False)
else:
    all_trial_df = pd.DataFrame()

print("\n=== Summary saved ===", flush=True)
print("SUMMARY_CSV:", SUMMARY_CSV, flush=True)
print("ALL_TRIAL_CSV:", ALL_TRIAL_CSV, flush=True)
print("n_summary_rows:", len(results_fixed_k_df), flush=True)
print("n_trial_rows:", len(all_trial_df), flush=True)

## summary

In [ ]:
summary_df = load_summary_or_empty()

if summary_df.empty:
    print("No summary file found.", flush=True)
else:
    print("\n=== Main paths ===", flush=True)
    print_main_paths()

    print("\n=== RMSE summary (standardized scale) ===", flush=True)
    for split in ["full", "train", "val", "test"]:
        print(
            f"{split:>5} | "
            f"stdk = {fmt_pm(summary_df[f'stdk_rmse_{split}'])} | "
            f"unreg = {fmt_pm(summary_df[f'unreg_rmse_{split}'])} | "
            f"reg_best = {fmt_pm(summary_df[f'reg_best_rmse_{split}'])}",
            flush=True,
        )

    print("\n=== RMSE summary (raw scale) ===", flush=True)
    for split in ["full", "train", "val", "test"]:
        print(
            f"{split:>5} | "
            f"stdk = {fmt_pm(summary_df[f'stdk_rmse_{split}_raw'])} | "
            f"unreg = {fmt_pm(summary_df[f'unreg_rmse_{split}_raw'])} | "
            f"reg_best = {fmt_pm(summary_df[f'reg_best_rmse_{split}_raw'])}",
            flush=True,
        )

    print("\n=== CovFrob summary (standardized scale) ===", flush=True)
    for split in ["full", "train", "val", "test"]:
        print(
            f"{split:>5} | "
            f"stdk = {fmt_pm(summary_df[f'stdk_covfrob_{split}'])} | "
            f"unreg = {fmt_pm(summary_df[f'unreg_covfrob_{split}'])} | "
            f"reg_best = {fmt_pm(summary_df[f'reg_best_covfrob_{split}'])}",
            flush=True,
        )

    print("\n=== CovFrob summary (raw scale) ===", flush=True)
    for split in ["full", "train", "val", "test"]:
        print(
            f"{split:>5} | "
            f"stdk = {fmt_pm(summary_df[f'stdk_covfrob_{split}_raw'])} | "
            f"unreg = {fmt_pm(summary_df[f'unreg_covfrob_{split}_raw'])} | "
            f"reg_best = {fmt_pm(summary_df[f'reg_best_covfrob_{split}_raw'])}",
            flush=True,
        )

    print("\n=== Semivariogram matching loss summary (standardized scale) ===", flush=True)
    for split in ["full", "train", "val", "test"]:
        print(
            f"{split:>5} | "
            f"stdk = {fmt_pm(summary_df[f'stdk_sv_loss_{split}'])} | "
            f"unreg = {fmt_pm(summary_df[f'unreg_sv_loss_{split}'])} | "
            f"reg_best = {fmt_pm(summary_df[f'reg_best_sv_loss_{split}'])}",
            flush=True,
        )

    print("\n=== Semivariogram matching loss summary (raw scale) ===", flush=True)
    for split in ["full", "train", "val", "test"]:
        print(
            f"{split:>5} | "
            f"stdk = {fmt_pm(summary_df[f'stdk_sv_loss_{split}_raw'])} | "
            f"unreg = {fmt_pm(summary_df[f'unreg_sv_loss_{split}_raw'])} | "
            f"reg_best = {fmt_pm(summary_df[f'reg_best_sv_loss_{split}_raw'])}",
            flush=True,
        )

    print("\n=== Best tuning summary ===", flush=True)
    print(f"best_tau1     : {fmt_pm(summary_df['best_tau1'])}", flush=True)
    print(f"best_tau2     : {fmt_pm(summary_df['best_tau2'])}", flush=True)
    print(f"{objective_label():<14}: {fmt_pm(summary_df[objective_label()])}", flush=True)

In [ ]:
# MAE / R^2 backfill from saved npz
def compute_metrics_for_split(y_true, y_pred, time_idx):
    yt = np.asarray(y_true[time_idx, :], dtype=np.float64)
    yp = np.asarray(y_pred[time_idx, :], dtype=np.float64)
    mask = np.isfinite(yt) & np.isfinite(yp)

    return {
        "mae": mae_pooled(yt, yp, mask),
        "r2": r2_pooled(yt, yp, mask),
    }


pred_files = list_prediction_npz_files()
summary_df = load_summary_or_empty()

if summary_df.empty:
    print("No summary file found.", flush=True)
elif len(pred_files) == 0:
    print("No prediction npz files found.", flush=True)
else:
    backfill_rows = []

    for path in pred_files:
        npz = np.load(path)

        seed_val = int(np.asarray(npz["seed"]).reshape(-1)[0])

        y_true_std = np.asarray(npz["y_true_run"], dtype=np.float64)
        y_true_raw = np.asarray(npz["y_true_raw_run"], dtype=np.float64)

        pred_map_std = {
            "stdk": np.asarray(npz["y_stdk_run"], dtype=np.float64),
            "unreg": np.asarray(npz["y_final_unreg_run"], dtype=np.float64),
            "reg_best": np.asarray(npz["y_final_reg_best_run"], dtype=np.float64),
        }

        pred_map_raw = {
            "stdk": np.asarray(npz["y_stdk_raw_run"], dtype=np.float64),
            "unreg": np.asarray(npz["y_final_unreg_raw_run"], dtype=np.float64),
            "reg_best": np.asarray(npz["y_final_reg_best_raw_run"], dtype=np.float64),
        }

        split_idx = {
            "full": np.arange(y_true_std.shape[0], dtype=int),
            "train": np.asarray(npz["train_time_idx_run"], dtype=int),
            "val": np.asarray(npz["val_time_idx_run"], dtype=int),
            "test": np.asarray(npz["test_time_idx_run"], dtype=int),
        }

        row = {"seed": seed_val}

        for model_name in ["stdk", "unreg", "reg_best"]:
            for split in ["full", "train", "val", "test"]:
                m_std = compute_metrics_for_split(
                    y_true_std,
                    pred_map_std[model_name],
                    split_idx[split],
                )
                m_raw = compute_metrics_for_split(
                    y_true_raw,
                    pred_map_raw[model_name],
                    split_idx[split],
                )

                row[f"{model_name}_mae_{split}"] = m_std["mae"]
                row[f"{model_name}_r2_{split}"] = m_std["r2"]
                row[f"{model_name}_mae_{split}_raw"] = m_raw["mae"]
                row[f"{model_name}_r2_{split}_raw"] = m_raw["r2"]

        backfill_rows.append(row)

    backfill_df = pd.DataFrame(backfill_rows)

    summary_df = summary_df.drop(
        columns=[
            c for c in backfill_df.columns
            if c != "seed" and c in summary_df.columns
        ],
        errors="ignore",
    )

    summary_df = summary_df.merge(backfill_df, on="seed", how="left")
    summary_df.to_csv(SUMMARY_CSV, index=False)

    print("\n=== MAE / R^2 backfill saved ===", flush=True)
    print("SUMMARY_CSV:", SUMMARY_CSV, flush=True)

    print("\n=== MAE summary (raw scale) ===", flush=True)
    for split in ["full", "train", "val", "test"]:
        print(
            f"{split:>5} | "
            f"stdk = {fmt_pm(summary_df[f'stdk_mae_{split}_raw'])} | "
            f"unreg = {fmt_pm(summary_df[f'unreg_mae_{split}_raw'])} | "
            f"reg_best = {fmt_pm(summary_df[f'reg_best_mae_{split}_raw'])}",
            flush=True,
        )

    print("\n=== R^2 summary (raw scale) ===", flush=True)
    for split in ["full", "train", "val", "test"]:
        print(
            f"{split:>5} | "
            f"stdk = {fmt_pm(100 * summary_df[f'stdk_r2_{split}_raw'])} | "
            f"unreg = {fmt_pm(100 * summary_df[f'unreg_r2_{split}_raw'])} | "
            f"reg_best = {fmt_pm(100 * summary_df[f'reg_best_r2_{split}_raw'])}",
            flush=True,
        )

## heat map

In [ ]:
# all_trial_df = load_all_trials_or_empty()

# if all_trial_df.empty:
#     print("No trial results file found.", flush=True)
# else:
#     heatmap_dir = DIAG_DIR / "heat_map"
#     heatmap_dir.mkdir(parents=True, exist_ok=True)

#     plot_trial_maps(
#         df=all_trial_df,
#         output_dir=heatmap_dir,
#         title_suffix=(
#             f"weather2k | target={TUNING_TARGET} | "
#             f"var={TARGET_VAR_IDX} | t_keep={T_KEEP} | k={K_FIXED}"
#         ),
#         file_prefix=(
#             f"weather2k_var{TARGET_VAR_IDX}"
#             f"_tkeep{T_KEEP}"
#             f"_k{K_FIXED}"
#             f"_{TUNING_TARGET}"
#         ),
#         best_by=objective_best_by_column(),
#         plots_to_draw=[
#             ("val_rmse_raw", "Validation RMSE (raw)"),
#             ("sv_loss_val", "Validation Semivariogram Matching Loss"),
#             ("covfrob_reg_val", "CovFrob (Val)"),
#             ("covfrob_reg_test", "CovFrob (Test)"),
#             ("smooth_penalty_per_entry_train", "Smooth Penalty / Entry (Train)"),
#             ("l1_penalty_per_entry_train", "L1 Penalty / Entry (Train)"),
#             ("smooth_over_recon_train", "Smooth / Recon (Train)"),
#             ("l1_over_recon_train", "L1 / Recon (Train)"),
#         ],
        
#     )

#     print("\n=== Heat maps saved ===", flush=True)
#     print("heatmap_dir:", heatmap_dir, flush=True)

## diagnostics

In [ ]:
# import matplotlib.pyplot as plt

# pred_files = list_prediction_npz_files()

# if len(pred_files) == 0:
#     print("No prediction files found.", flush=True)
# else:
#     eig_dir = DIAG_DIR / "eigen_structure"
#     eig_dir.mkdir(parents=True, exist_ok=True)

#     def empirical_cov_from_time_field(y):
#         y = np.asarray(y, dtype=np.float64)
#         if y.ndim != 2:
#             raise ValueError(f"y must be 2D, got shape={y.shape}")
#         if y.shape[0] < 2:
#             return np.full((y.shape[1], y.shape[1]), np.nan, dtype=np.float64)
#         yc = y - np.mean(y, axis=0, keepdims=True)
#         cov = (yc.T @ yc) / (y.shape[0] - 1)
#         cov = 0.5 * (cov + cov.T)
#         return cov

#     def eig_desc(cov):
#         vals, vecs = np.linalg.eigh(cov)
#         order = np.argsort(vals)[::-1]
#         vals = vals[order]
#         vecs = vecs[:, order]
#         vals = np.maximum(vals, 0.0)
#         return vals, vecs

#     def cumulative_ratio(vals):
#         vals = np.asarray(vals, dtype=np.float64)
#         total = np.sum(vals)
#         if total <= 0:
#             return np.full_like(vals, np.nan, dtype=np.float64)
#         return np.cumsum(vals) / total

#     def topk_energy_ratio(vals, k):
#         vals = np.asarray(vals, dtype=np.float64)
#         total = np.sum(vals)
#         if total <= 0:
#             return float("nan")
#         k = min(k, len(vals))
#         return float(np.sum(vals[:k]) / total)

#     def eigenvector_alignment(vec_ref, vec_cmp):
#         vec_ref = np.asarray(vec_ref, dtype=np.float64).reshape(-1)
#         vec_cmp = np.asarray(vec_cmp, dtype=np.float64).reshape(-1)
#         denom = np.linalg.norm(vec_ref) * np.linalg.norm(vec_cmp)
#         if denom <= 1e-12:
#             return float("nan")
#         return float(np.abs(np.dot(vec_ref, vec_cmp)) / denom)

#     def load_split_arrays(npz_obj, split):
#         if split == "full":
#             idx = np.arange(npz_obj["y_true_run"].shape[0], dtype=int)
#         elif split == "train":
#             idx = np.asarray(npz_obj["train_time_idx_run"], dtype=int)
#         elif split == "val":
#             idx = np.asarray(npz_obj["val_time_idx_run"], dtype=int)
#         elif split == "test":
#             idx = np.asarray(npz_obj["test_time_idx_run"], dtype=int)
#         else:
#             raise ValueError(f"Unknown split: {split}")

#         y_true = np.asarray(npz_obj["y_true_run"], dtype=np.float64)[idx, :]
#         y_stdk = np.asarray(npz_obj["y_stdk_run"], dtype=np.float64)[idx, :]
#         y_unreg = np.asarray(npz_obj["y_final_unreg_run"], dtype=np.float64)[idx, :]
#         y_reg = np.asarray(npz_obj["y_final_reg_best_run"], dtype=np.float64)[idx, :]
#         return y_true, y_stdk, y_unreg, y_reg

#     split_list = ["val", "test"]
#     top_k_list = [1, 3, 5, 10]

#     eig_rows = []
#     align_rows = []
#     spectrum_store = {split: {"obs": [], "stdk": [], "unreg": [], "reg_best": []} for split in split_list}

#     for path in pred_files:
#         npz = np.load(path)
#         seed_val = int(np.asarray(npz["seed"]).reshape(-1)[0])

#         for split in split_list:
#             y_obs, y_stdk, y_unreg, y_reg = load_split_arrays(npz, split)

#             cov_obs = empirical_cov_from_time_field(y_obs)
#             cov_stdk = empirical_cov_from_time_field(y_stdk)
#             cov_unreg = empirical_cov_from_time_field(y_unreg)
#             cov_reg = empirical_cov_from_time_field(y_reg)

#             eig_obs, vec_obs = eig_desc(cov_obs)
#             eig_stdk, vec_stdk = eig_desc(cov_stdk)
#             eig_unreg, vec_unreg = eig_desc(cov_unreg)
#             eig_reg, vec_reg = eig_desc(cov_reg)

#             spectrum_store[split]["obs"].append(eig_obs)
#             spectrum_store[split]["stdk"].append(eig_stdk)
#             spectrum_store[split]["unreg"].append(eig_unreg)
#             spectrum_store[split]["reg_best"].append(eig_reg)

#             max_k = min(10, len(eig_obs))
#             cum_obs = cumulative_ratio(eig_obs)
#             cum_stdk = cumulative_ratio(eig_stdk)
#             cum_unreg = cumulative_ratio(eig_unreg)
#             cum_reg = cumulative_ratio(eig_reg)

#             for k in range(1, max_k + 1):
#                 eig_rows.append({
#                     "seed": seed_val,
#                     "split": split,
#                     "k": k,
#                     "obs_eig": float(eig_obs[k - 1]),
#                     "stdk_eig": float(eig_stdk[k - 1]),
#                     "unreg_eig": float(eig_unreg[k - 1]),
#                     "reg_best_eig": float(eig_reg[k - 1]),
#                     "obs_cum_ratio": float(cum_obs[k - 1]),
#                     "stdk_cum_ratio": float(cum_stdk[k - 1]),
#                     "unreg_cum_ratio": float(cum_unreg[k - 1]),
#                     "reg_best_cum_ratio": float(cum_reg[k - 1]),
#                 })

#             for k in top_k_list:
#                 kk = min(k, len(eig_obs))
#                 align_rows.append({
#                     "seed": seed_val,
#                     "split": split,
#                     "top_k": kk,
#                     "obs_topk_energy": float(topk_energy_ratio(eig_obs, kk)),
#                     "stdk_topk_energy": float(topk_energy_ratio(eig_stdk, kk)),
#                     "unreg_topk_energy": float(topk_energy_ratio(eig_unreg, kk)),
#                     "reg_best_topk_energy": float(topk_energy_ratio(eig_reg, kk)),
#                 })

#             max_align_k = min(5, vec_obs.shape[1], vec_stdk.shape[1], vec_unreg.shape[1], vec_reg.shape[1])
#             for k in range(max_align_k):
#                 align_rows.append({
#                     "seed": seed_val,
#                     "split": split,
#                     "top_k": -(k + 1),
#                     "obs_topk_energy": np.nan,
#                     "stdk_topk_energy": eigenvector_alignment(vec_obs[:, k], vec_stdk[:, k]),
#                     "unreg_topk_energy": eigenvector_alignment(vec_obs[:, k], vec_unreg[:, k]),
#                     "reg_best_topk_energy": eigenvector_alignment(vec_obs[:, k], vec_reg[:, k]),
#                 })

#     eig_df = pd.DataFrame(eig_rows)
#     align_df = pd.DataFrame(align_rows)

#     eig_df.to_csv(eig_dir / "eigen_spectrum_summary.csv", index=False)
#     align_df.to_csv(eig_dir / "eigen_alignment_summary.csv", index=False)

#     for split in split_list:
#         plot_df = eig_df[eig_df["split"] == split].copy()
#         if plot_df.empty:
#             continue

#         mean_df = (
#             plot_df.groupby("k", as_index=False)[
#                 [
#                     "obs_eig",
#                     "stdk_eig",
#                     "unreg_eig",
#                     "reg_best_eig",
#                     "obs_cum_ratio",
#                     "stdk_cum_ratio",
#                     "unreg_cum_ratio",
#                     "reg_best_cum_ratio",
#                 ]
#             ]
#             .mean()
#         )

#         plt.figure(figsize=(7, 5))
#         plt.plot(mean_df["k"], mean_df["obs_eig"], label="Observed")
#         plt.plot(mean_df["k"], mean_df["stdk_eig"], label="STDK")
#         plt.plot(mean_df["k"], mean_df["unreg_eig"], label="Unreg")
#         plt.plot(mean_df["k"], mean_df["reg_best_eig"], label="Reg Best")
#         plt.xlabel("Eigenvalue rank")
#         plt.ylabel("Mean eigenvalue")
#         plt.title(f"Eigen spectrum | {split}")
#         plt.legend()
#         plt.tight_layout()
#         plt.savefig(eig_dir / f"eigen_spectrum_{split}.png", dpi=200)
#         plt.close()

#         plt.figure(figsize=(7, 5))
#         plt.plot(mean_df["k"], mean_df["obs_cum_ratio"], label="Observed")
#         plt.plot(mean_df["k"], mean_df["stdk_cum_ratio"], label="STDK")
#         plt.plot(mean_df["k"], mean_df["unreg_cum_ratio"], label="Unreg")
#         plt.plot(mean_df["k"], mean_df["reg_best_cum_ratio"], label="Reg Best")
#         plt.xlabel("Top-k")
#         plt.ylabel("Cumulative explained variance ratio")
#         plt.title(f"Cumulative eigen energy | {split}")
#         plt.legend()
#         plt.tight_layout()
#         plt.savefig(eig_dir / f"eigen_cumulative_ratio_{split}.png", dpi=200)
#         plt.close()

#     energy_df = align_df[align_df["top_k"] > 0].copy()
#     vec_align_df = align_df[align_df["top_k"] < 0].copy()

#     if not energy_df.empty:
#         energy_print = (
#             energy_df.groupby(["split", "top_k"])[
#                 ["obs_topk_energy", "stdk_topk_energy", "unreg_topk_energy", "reg_best_topk_energy"]
#             ]
#             .mean()
#             .reset_index()
#         )

#         print("\n=== Top-k energy ratio summary ===", flush=True)
#         for split in split_list:
#             sub = energy_print[energy_print["split"] == split]
#             for _, row in sub.iterrows():
#                 k = int(row["top_k"])
#                 print(
#                     f"{split:>4} | top-{k} | "
#                     f"obs = {row['obs_topk_energy']:.6f} | "
#                     f"stdk = {row['stdk_topk_energy']:.6f} | "
#                     f"unreg = {row['unreg_topk_energy']:.6f} | "
#                     f"reg_best = {row['reg_best_topk_energy']:.6f}",
#                     flush=True,
#                 )

#     if not vec_align_df.empty:
#         vec_align_print = (
#             vec_align_df.assign(eig_rank=lambda d: -d["top_k"])
#             .groupby(["split", "eig_rank"])[
#                 ["stdk_topk_energy", "unreg_topk_energy", "reg_best_topk_energy"]
#             ]
#             .mean()
#             .reset_index()
#         )

#         print("\n=== Eigenvector alignment summary ===", flush=True)
#         for split in split_list:
#             sub = vec_align_print[vec_align_print["split"] == split]
#             for _, row in sub.iterrows():
#                 k = int(row["eig_rank"])
#                 print(
#                     f"{split:>4} | eigvec-{k} | "
#                     f"stdk = {row['stdk_topk_energy']:.6f} | "
#                     f"unreg = {row['unreg_topk_energy']:.6f} | "
#                     f"reg_best = {row['reg_best_topk_energy']:.6f}",
#                     flush=True,
#                 )

#     print("\n=== Eigen diagnostics saved ===", flush=True)
#     print("eig_dir:", eig_dir, flush=True)